# 11 — Error analysis (Phase 11)

**Goal:** understand the remaining mistakes well enough to choose the *next* improvement on evidence, not guesswork.

We analyse the **tuning** entities (train split) using their **out-of-fold** probabilities from Phase 9 —
so the validation set stays untouched for measuring the improvements.

Three kinds of mistakes (threshold 0.70):

| kind | what happened | costs |
|---|---|---|
| **A. false merge** | we predicted a pair that is a different business | precision (counts double in F0.5); a singleton drops to 0 |
| **B. missed candidate** | a true match was a candidate, but scored < 0.70 | recall |
| **C. missed by blocking** | a true match never became a candidate | recall — no model can fix it |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

from src.evaluation import score_per_entity
from src.decision import to_submission
from src.rule_experiment import load_query_sets

pd.set_option("display.width", 230)
pd.set_option("display.max_colwidth", 45)
THRESHOLD = 0.70
cache = PROJECT_ROOT / "cache"

features = pd.read_parquet(cache / "phase8_features.parquet")
features = features[features["query_set"] == "tuning"].reset_index(drop=True)
probs = pd.read_parquet(cache / "phase9_predictions.parquet")
probs = probs[probs["query_set"] == "tuning"][["source1_entity_id", "candidate_id", "probability"]]
pairs = features.merge(probs, on=["source1_entity_id", "candidate_id"])
pairs["predicted"] = pairs["probability"] >= THRESHOLD
_, truth = load_query_sets()
truth = truth["tuning"]
truth["n_true"] = truth["matched_entity_ids"].map(lambda s: len(s.split(",")) if s else 0)
pairs = pairs.merge(truth[["source1_entity_id", "n_true"]], on="source1_entity_id")
print(f"{len(pairs):,} candidate pairs of {truth.shape[0]:,} tuning entities")

998,797 candidate pairs of 20,000 tuning entities


## How much does each kind of mistake cost?

We count the mistakes, and estimate the **F0.5 lost** by each kind: re-score the tuning entities after
"fixing" only that kind of mistake (e.g. removing all false merges) and see how much the score would rise.

In [2]:
all_true = truth.assign(match_id=truth["matched_entity_ids"].str.split(",")).explode("match_id")
all_true = all_true[all_true["match_id"].notna() & (all_true["match_id"] != "")][["source1_entity_id", "match_id"]]
candidate_keys = set(zip(pairs["source1_entity_id"], pairs["candidate_id"]))
all_true["is_candidate"] = [(a, b) in candidate_keys for a, b in zip(all_true["source1_entity_id"], all_true["match_id"])]

current = pairs[pairs["predicted"]]
fix_a = current[current["label"] == 1]                                          # no false merges
fix_b = pd.concat([current, pairs[(pairs["label"] == 1) & ~pairs["predicted"]]])  # all candidate matches found
fix_c = pd.concat([current, all_true.loc[~all_true["is_candidate"]].rename(columns={"match_id": "candidate_id"})])

def macro(matches):
    return score_per_entity(to_submission(matches), truth)["f05"].mean()

now = macro(current)
cost = pd.DataFrame({
    "kind": ["A. false merges", "B. missed candidates", "C. missed by blocking"],
    "count (pairs)": [len(current) - len(fix_a), int(((pairs["label"] == 1) & ~pairs["predicted"]).sum()), int((~all_true["is_candidate"]).sum())],
    "F0.5 if fixed": [macro(fix_a), macro(fix_b), macro(fix_c)],
})
cost["F0.5 gain"] = cost["F0.5 if fixed"] - now
print(f"current tuning F0.5 (out-of-fold): {now:.4f}")
cost.round(4)

current tuning F0.5 (out-of-fold): 0.9324


,kind,count (pairs),F0.5 if fixed,F0.5 gain
0,A. false merges,1401,0.9501,0.0176
1,B. missed candidates,4853,0.9587,0.0262
2,C. missed by blocking,4203,0.9595,0.0270


**All three kinds cost about the same** (+0.018 to +0.027 each if fully fixed).
Blocking misses (C) are *not* fixable by any model, only by better candidate generation; A and B are about the model and features.
Note the asymmetry: 1,401 false merges cost almost as much as 4,853 missed candidates — precision counts double.

---
## A. False merges — categories

Each false merge gets the **first** category that fits (the order matters):

| category | rule |
|---|---|
| singleton wrongly matched | the S1 entity has no true match at all |
| same name, other address | names nearly identical (token set ≥ 0.9), addresses clearly different (token set < 0.7) |
| same address, other name | addresses nearly identical, names different (core token set < 0.6) |
| legal or number conflict | `legal_conflict` or `num_conflict` = 1 |
| near-duplicate | names **and** addresses both very similar |
| other | anything else |

In [3]:
def categorize_false_merges(fp):
    conditions = [
        fp["n_true"] == 0,
        (fp["name_token_set"] >= 0.9) & (fp["addr_token_set"].fillna(0) < 0.7),
        (fp["addr_token_set"].fillna(0) >= 0.9) & (fp["core_token_set"] < 0.6),
        (fp["legal_conflict"] == 1) | (fp["num_conflict"] == 1),
        (fp["name_token_set"] >= 0.9) & (fp["addr_token_set"].fillna(0) >= 0.9),
    ]
    names = ["singleton wrongly matched", "same name, other address", "same address, other name",
             "legal or number conflict", "near-duplicate"]
    return np.select(conditions, names, "other")

fp = pairs[pairs["predicted"] & (pairs["label"] == 0)].copy()
fp["category"] = categorize_false_merges(fp)
fp["category"].value_counts().to_frame("false merges")

,false merges
category,
other,446
"same address, other name",322
"same name, other address",228
near-duplicate,190
singleton wrongly matched,110
legal or number conflict,105


In [4]:
cols = ["name_s1", "name_c", "addr_s1", "addr_c", "probability", "name_token_set", "addr_token_set", "num_shared", "legal_conflict", "num_conflict"]
for category in fp["category"].value_counts().index:
    print(f"\n==== {category}")
    display(fp.loc[fp["category"] == category, cols].sample(min(5, (fp["category"] == category).sum()), random_state=0))


==== other


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
981795,tralion international llc,vioavi international llc,106 harvard avenue medford ma,106 harvard avenue medford massachusetts,0.964779,0.829268,0.945455,1.0,0,0
924984,dhairya traders pvt ltd,dhairya seats pvt ltd,no 181a and 181b kajjari grama ranebennur...,door no 183a and 181b haveri ಕರ್ನಾಟಕ,0.976301,0.833333,0.666667,1.0,0,0
702174,nannette hill sterling partners,hood hill sterling partners,4612 82nd terrace kansas city mo,4612 82nd ter kansas city mo,0.862431,0.897959,0.933333,2.0,0,0
913904,tio,peridosbancshares,1294 350th avenue burdett ks,1294 350rd avenue burdett kansas,0.721189,0.200000,0.866667,2.0,0,0
350209,drayos chenghe,drayova chenghe,889 park place iowa city ia,889 park place po box 3829 iowa city ia,0.913123,0.896552,1.000000,1.0,0,0



==== same address, other name


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
905140,red tech private limited,रियल एनर्जी लिमिटेड,hno 220 shutter vali gali south delhi kha...,delhi south delhi khanpur delhi south del...,0.848561,0.093023,1.000000,1.0,0,0
205258,madhuraj technology pvt ltd,सिटी फाउंडेशन प्राइवेट लिमिटेड,haryana sadar bazar 4 18 laxmi garden sec...,498 4 18 laxmi garden sector 11 gurgaon s...,0.946842,0.105263,0.921569,3.0,0,0
73508,silver developers private limited,सिल्वर डेवलपर्स स्टोर्स प्राइवेट लिमिटेड,jaiganeshsamrajya unitk1 rowhno2 punenash...,door no 31 jaiganeshsamrajya unitk1 rowhn...,0.941870,0.082192,1.000000,2.0,0,0
836531,gonzalez global inc,zephnyla,7115 bellhaven boulevard unit 80a charlot...,7115 bellhaven boulevard unit 80a charlot...,0.847797,0.222222,1.000000,2.0,0,0
434460,montance hills inc,montanke inc center,11319 773 ben wheeler tx,11319 773 ben wheeler tx,0.876050,0.648649,1.000000,2.0,0,0



==== same name, other address


...

,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
570650,krishna services private limited,kst private services limited,15 58 friends enclave flat no a 1 civil l...,15 58 1 friends enclave civil lines kanpu...,0.965202,0.923077,0.903226,3.0,0,0
373460,physical therapy specialists of jackson h...,physical therapy specialists of jackson h...,37 56 85 street jackson heights ny,37 59 85 street jackson heights new york,0.736487,1.000000,0.903226,2.0,0,0
838248,hundred traders private limited,m s hundred traders limited,c o arvind gali no 3 near rue p singh coa...,door no 25 c o arvind gali no 3 near rue ...,0.953012,0.920000,1.000000,1.0,0,0
500337,orbyn lionheart,orbny lionheart,233 ackerman road stevensville md,maryland stevensville 233 ackerman road,0.989898,0.933333,0.952381,1.0,0,0
669537,novon inflection llc,novoix inflection llc,247 union street unit 102a athens oh,oh 247 union street athens,0.958430,0.926829,1.000000,1.0,0,0



==== singleton wrongly matched


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
785353,liis corporation,liis corporation llc,5128 harvard avenue tulsa ok,5129 harvard avenue tulsa ok,0.817285,1.000000,0.964286,0.0,0,1
60841,juno india makers pvt ltd,dr juno india makers llp,3rd to 5th floor hudaa citycentre metrost...,no 5rd to 5th floor hudaa citycentre metr...,0.758151,0.857143,0.911765,1.0,1,0
667542,evers quartzsea,evers quartzsea llc,204 market street oh tiffin,,0.921035,1.000000,NaN,0.0,0,0
1093,eagle plastics private limited,সূর্য কনসালটেন্সি প্রাইভেট লিমিটেড,jafala road p o inda kharagpur dist pasch...,jafala road p o inda kharagpur dist pasch...,0.891407,0.093750,0.919708,0.0,0,0
174140,global enterprises private limited,ग्लोबल सॉल्यूशंस प्राइवेट लिमिटेड,e 63 basement sector 11 near ymca faridab...,e 63 basement sector 14 near ymca faridab...,0.728580,0.089552,0.930233,1.0,0,0



==== legal or number conflict


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
277167,371 372 east owners corp,371 372 east owners corp corp,12438 beaver avenue victorville ca,12441 beaver avenue victorville california,0.904170,1.000000,0.847458,0.0,0,1
610919,pioneer solutions private limited,pioneer solutions private ltd,plot no 8 lahariya park yashoda nagar kan...,181 c k block yashoda nagar kanpur uttar ...,0.878822,0.935484,0.829268,0.0,0,1
20601,makson infra of nagpur,makson infra of nagpur limited,nagpur panchyanshi plot area plot no 15 n...,plot no 16 panchyanshi plot area nagpur म...,0.763645,1.000000,0.815789,0.0,0,1
621805,lm safe allspring inc,lmv safe allspring inc,213 clements avenue danville il,215 clements avenue danville illinois,0.888457,0.976744,0.882353,0.0,0,1
817989,sioux t torres dmd group,dmz t torres sioux group,tn 10640 dutchtown road concord,10649 dutchtown road concord tennessee,0.889277,0.958333,0.869565,0.0,0,1


**What the false merges tell us**
- **Same address, other name (322)** — many are an English S1 name vs an **Indian-script** candidate name at the same address
  (`red tech private limited` vs `रियल एनर्जी लिमिटेड` = "Real Energy Limited"). The model cannot read the Hindi name, so it trusts
  the address — and the address belongs to a *different* business. Others are completely different names at the same address.
- **Same name, other address (228)** — mostly candidates with **no address** and the same (often generic) name:
  there is no evidence to separate the right entity from its look-alikes.
- **Near-duplicates (190)** — names and addresses nearly identical (`orbyn lionheart` vs `orbny lionheart`, same street):
  the ground truth says "different business". Some of these may be irreducible or label noise.
- **Singletons wrongly matched (110)** and **legal/number conflicts (105)** — e.g. `12438` vs `12441 beaver avenue`:
  the conflict features fired, but the rest of the evidence outweighed them.
- **Other (446)** — shared generic words + same street, different distinctive word (`tralion international llc` vs `vioavi international llc`).

---
## B. Missed candidates (true match, probability < 0.70) — categories

In [5]:
def categorize_misses(fn):
    website = fn["name_c"].str.fullmatch(r"\S{12,}") & (fn["name_s1"].str.count(" ") >= 1)
    conditions = [
        fn["name_c_foreign_script"] == 1,
        website,
        fn["addr_c_empty"] == 1,
        fn["core_token_set"] < 0.5,
        fn["addr_token_set"].fillna(0) < 0.6,
        fn["probability"] >= 0.3,
    ]
    names = ["name in Indian script", "website / glued name", "candidate has no address",
             "different name (trade name?)", "address very different", "borderline (0.3-0.7)"]
    return np.select(conditions, names, "other")

fn = pairs[~pairs["predicted"] & (pairs["label"] == 1)].copy()
fn["category"] = categorize_misses(fn)
fn.groupby("category").agg(missed=("probability", "size"), mean_probability=("probability", "mean")).sort_values("missed", ascending=False).round(3)

,missed,mean_probability
category,,
borderline (0.3-0.7),1169,0.510
candidate has no address,1042,0.368
other,1005,0.132
name in Indian script,731,0.330
different name (trade name?),721,0.246
website / glued name,120,0.259
address very different,65,0.178


In [6]:
for category in fn["category"].value_counts().index:
    print(f"\n==== {category}")
    display(fn.loc[fn["category"] == category, cols].sample(min(5, (fn["category"] == category).sum()), random_state=0))


==== borderline (0.3-0.7)


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
753997,a trusted wellness,a trusted wellness lp,unit i295 or 1497 brookside avenue dallas,497 brookside avenue dallas or,0.472790,1.000000,0.928571,0.0,0,1
722494,estrellita chin colonial money,estrellita chin,littlefield 781 chandler drive az,794 chandler drive littlefield az,0.401256,1.000000,0.939394,0.0,0,1
476601,zen management limited,zen mnagmseet limited,h no 63 a 2 a defence colony near cod agr...,उत्तर प्रदेश 172 a 2 a dnfence colony nea...,0.521902,0.883721,0.660194,1.0,0,0
288983,solar resorts private limited,solar resorts limited limited service,c 32 south extension 2 ndse 2 delhi south...,door no 64 c 32 south extension 2 ndse 2 ...,0.519127,0.862069,1.000000,2.0,0,0
358047,neon marketing,mr neon markteibg,sevashram society near atma jyoti ashram ...,244 a 2 sevashram society near atma jyoti...,0.476416,0.774194,0.969697,1.0,0,0



==== candidate has no address


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
730201,construction tegh consultancy,construction tegh,11 2 89 11 2 111 komati galli ward no 25 ...,,0.164224,1.00,NaN,0.0,0,0
197333,kaverendra health of ahmedabad pvt ltd,kaverendra health of ahmedabad pvt,812 1 to 4 amrakunj farm opp honest parin...,,0.645161,1.00,NaN,0.0,0,0
861034,sioux falls sterling microelectronics,sioux fhrlls sterling microelectronics,2709 28th street sioux falls sd,,0.488836,0.96,NaN,0.0,0,0
591928,evans and fortenberry lp,evans and fortenberry,1630 park avenue shawnee ok,,0.676231,1.00,NaN,0.0,0,0
612806,luevano ottman riggs bitcoin llc,luevano ottman riggs bitcoin llc,101 102 nina drive virginia beach city va,,0.687314,1.00,NaN,0.0,0,0



==== other


...

,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
699296,green land private limited,nylasyn,no 23 24 25 arul murugan nagar palamthand...,kanchipuram 24 25 arul murugan nagar pala...,0.591876,0.242424,1.000000,3.0,0,0
892456,twisted tattoo,deltajax,197 baptist church road eagleville tn,197 baptist churhc road eagleville tennessee,0.668463,0.181818,0.888889,1.0,0,0
334848,peridan river holdings,lyrabrix,3814 windmill court celina tx,3814 windmill court celina tx,0.203609,0.333333,1.000000,1.0,0,0
197033,heard international llc,jaxmira,1995 carson drive beaumont tx,1995 carson drive beaumont texas,0.406629,0.266667,0.950820,1.0,0,0
686683,nk rehabilitation,nk center,fl a1 88 motilal colony milan pally sulta...,fl a1 88 motilal colohy milan pally sulta...,0.132283,0.384615,0.951724,3.0,0,0



==== website / glued name


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
386853,ballard bright muzero,muzerobrightballard,61 park drive brookhaven ny,rocky point 61 park drive new york,0.347878,0.400000,0.650000,1.0,0,0
777580,lyray clean inc,vantagelumd0va,601 peyton street lynchburg city va,601 peyton street lynchburg va,0.123964,0.275862,1.000000,1.0,0,0
780228,ghar welfare society,dovabelodova,c o krishna prasad ward 4 kandhwaliya bag...,no 134 c o krishna prasad ward 4 kandhwal...,0.045355,0.250000,0.885496,1.0,0,0
59727,reix llc,belotavodelta,1115 wood street statesville nc,1115 wood street statesville nc,0.011177,0.190476,1.000000,1.0,0,0
785747,siracusa mining inc,dovadrextavo,14621 ava leigh avenue city of el paso tx,14621 ava leigh avenue city of el paso texas,0.668388,0.129032,0.964706,1.0,0,0



==== address very different


,name_s1,name_c,addr_s1,addr_c,probability,name_token_set,addr_token_set,num_shared,legal_conflict,num_conflict
731975,rds academy ltd,rds academy,7 periyakinaru street ammapet salem tamil...,no 555 3 7 salem tn,0.324947,1.0,0.538462,1.0,0,0
532274,sai india food private limited,sai india food,shop no 1 gat no 1003 nigade nagar phalta...,shop no 1 shirwal dist khandala mh,0.083762,1.0,0.500000,1.0,0,0
718274,continental institute p c,continental institute p c,1195 privet drive unit 316 cottonwood hei...,8195 privet drive pmb 2856 salt lake city ut,0.154699,1.0,0.565217,0.0,0,1
961763,al software pvt ltd,al software pvt ltd,28 8 1 c n roy road kolkata howrah west b...,58 8 1 wb,0.374292,1.0,0.500000,2.0,0,0
622488,anand investment private limited,anand investment private limited,iceri 9 3rd block 21st cross 4th stage 8t...,122 icri 9 bangalore ಕರ್ನಾಟಕ,0.001133,1.0,0.564103,1.0,0,0


**What the missed candidates tell us**
- **Borderline (1,169, mean p = 0.51)** — the model was unsure; more training data and better features move these.
- **No address (1,042)** — only the name is available, and generic names have many look-alikes, so the model is (rightly) cautious.
- **Indian-script name (731)** — the name features see 0 similarity; only the address carries evidence.
- **Different name / trade name (721)** — invented brand names (`nylasyn`, `deltajax`) at the same address. This is the mirror image
  of the "same address, other name" false merges: same address + unrelated name is *sometimes* the same business, sometimes not.
- **Website / glued names (120)** — mostly invented glued names (`dovadrextavo`), not the S1 name joined.

---
## C. Missed by blocking — what do these true matches look like?

In [7]:
queries = pd.read_parquet(cache / "phase7_queries.parquet")
missed = all_true[~all_true["is_candidate"]]
missed_text = pd.read_parquet(cache / "train_pool_norm.parquet", filters=[("entity_id", "in", list(missed["match_id"]))])
missed = (missed.merge(queries[["entity_id", "name_norm", "addr_norm"]], left_on="source1_entity_id", right_on="entity_id")
                .drop(columns="entity_id")
                .merge(missed_text, left_on="match_id", right_on="entity_id", suffixes=("_s1", "_c")).drop(columns="entity_id"))
missed["name_token_set"] = process.cpdist(missed["name_norm_s1"].tolist(), missed["name_norm_c"].tolist(), scorer=fuzz.token_set_ratio, workers=-1) / 100
missed["addr_token_set"] = process.cpdist(missed["addr_norm_s1"].tolist(), missed["addr_norm_c"].tolist(), scorer=fuzz.token_set_ratio, workers=-1) / 100

conditions = [
    missed["name_norm_c"].str.contains("[\u0900-\u0d7f]") & (missed["addr_norm_c"] == ""),
    missed["name_norm_c"].str.contains("[\u0900-\u0d7f]"),
    missed["addr_norm_c"] == "",
    (missed["name_token_set"] >= 0.8) & (missed["addr_token_set"] >= 0.8),
]
names = ["Indian-script name AND no address", "Indian-script name", "no address", "similar name + address (lost in ranking)"]
missed["category"] = np.select(conditions, names, "other")
print(f"true matches missed by blocking: {len(missed):,}")
missed["category"].value_counts().to_frame("missed by blocking")

true matches missed by blocking: 4,203


,missed by blocking
category,
Indian-script name,1184
other,1036
similar name + address (lost in ranking),1014
no address,961
Indian-script name AND no address,8


In [8]:
for category in missed["category"].value_counts().index:
    print(f"\n==== {category}")
    display(missed.loc[missed["category"] == category, ["name_norm_s1", "name_norm_c", "addr_norm_s1", "addr_norm_c", "name_token_set", "addr_token_set"]]
            .sample(min(5, (missed["category"] == category).sum()), random_state=0))


==== Indian-script name


,name_norm_s1,name_norm_c,addr_norm_s1,addr_norm_c,name_token_set,addr_token_set
2823,krishna international pvt ltd,ಕೃಷ್ಣ ಇಂಟರ್ನ್ಯಾಷನಲ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,no a 18 ground floor sterling park kodige...,no a d 18 bengaluru bangalore karnataka,0.088235,0.973684
1875,tirupati services pvt ltd,திருப்பதி சர்வீசஸ் பிரைவேட் லிமிடெட்,51 2 1st floor 11th avenue ashok nagar ch...,51 2 chennai tamil nadu,0.098361,1.000000
3255,ram shivam ventures limited,ರಾಮ್ ಶಿವಂ ವೆಂಚರ್ಸ್ ಲಿಮಿಟೆಡ್,97 1 east anjaneya temple street bangalor...,97 1 n a bangalore south bangalore karnataka,0.111111,0.937500
3173,jain surya care private limited,জৈন সূর্য কেয়ার প্রাইভেট লিমিটেড,1a old post office street near kolkata hi...,plot 559 1a howrah calcutta west bengal,0.125000,0.700000
1146,international marketing,ಇಂಟರ್ನ್ಯಾಷನಲ್ ಮಾರ್ಕೆಟಿಂಗ್,748 1st stage sathagally layout mysore ka...,g 748 mysore karnataka,0.041667,0.952381



==== other


,name_norm_s1,name_norm_c,addr_norm_s1,addr_norm_c,name_token_set,addr_token_set
2748,st fellowship,fst,1822 mayfair drive city of appleton wi,mayfair drive city of appleton wisconsin,0.375000,0.882353
1477,pranjal brothers private limited,pranjalbrotherscom,n 3 ndse 1 ring road new delhi south delh...,n 3 south delhi new delhi dl,0.400000,0.926829
376,prime properties private limited,nylacalo,shop no 701 mahalaxmi complex h no 729 ud...,701 pune महाराष्ट्र,0.150000,0.592593
293,ashapura trading llp,ashapura llp center,ayodhya 11th a main road nri layout 2mhmw...,ayodhca bangalore north bangalore ಕರ್ನಾಟಕ,0.774194,0.652174
990,anjani solutions pvt ltd,anjani pvt ltd service 2221958781,1 2c floor 1 7 kulsum terrace walton road...,mumbai region mh mumbai city 3 2c,0.736842,0.700000



==== similar name + address (lost in ranking)


,name_norm_s1,name_norm_c,addr_norm_s1,addr_norm_c,name_token_set,addr_token_set
3863,5 legged dog llc,5 efgged dog llc,oh delta 702 hickory lane,hickory lane delta oh,0.937500,1.000000
411,cascade annaly p c,cascade anna1y p c,2201 17th avenue hutchinson ks,17th avenue hutchinson ks,0.944444,1.000000
2580,arihant alpha foods private limited,arihant alpha f0ods private limited,1511 rg trade tower netaji subhash place ...,1511 delhi north west delhi दिल्ली,0.971429,0.857143
2621,vinson india private limited,vinson private limited services services,shop no b 7 s no 48 11 to 48 1 chavan ind...,pune shop no b 7 mh pune city,0.880000,0.933333
1693,internal medicine health,internal medicine,7347 woodward avenue unit 112 woodridge il,7347d woodward avenue woodridge city illi...,1.000000,0.804598



==== no address


,name_norm_s1,name_norm_c,addr_norm_s1,addr_norm_c,name_token_set,addr_token_set
197,classic robotics technologies inc,classic robotics enterprises,111 82 road kew gardens ny,,0.754098,0.0
3788,ear nose throat rocky medicine,ear nose throat rocky center,1024 rocky lane irving tx,,0.862069,0.0
51,prairie direct mission inc,prairie mission inc services,3702 clarence avenue berwyn il,,0.844444,0.0
2491,oncology pinnacle associates p c,oncology pinnacle associates inc,4250 journeys end road spotsylvania count...,,0.937500,0.0
561,highland ventures inc,high1and inc ventures,31 morris avenue athens oh,,0.952381,0.0



==== Indian-script name AND no address


,name_norm_s1,name_norm_c,addr_norm_s1,addr_norm_c,name_token_set,addr_token_set
3017,supreme systems private limited,supreme सिस्टम्स प्राइवेट लिमिटेड,laxmi market rewa road satna madhya pradesh,,0.368421,0.0
2187,jain foods private limited,jain foods প্রাইভেট লিমিটেড,21 d subhash pally kolkata kolkata howrah...,,0.555556,0.0
795,bharat energy limited,bharat एनर्जी लिमिटेड,at post h no 667 1 nagaon dhule dhule mah...,,0.444444,0.0
4059,star impex pvt ltd,star impex ప్రైవేట్ లిమిటెడ్,hyderabad rajarajeswari nag kondapu rue t...,,0.714286,0.0
2417,shakti technologies,shakti టెక్నాలజీస్,h no 10 1 72 mamillagudem khammam telangana,,0.500000,0.0


**What blocking misses**
- **Indian-script name (1,184)** — the biggest group: the name keys never match, and the address keys alone were not rare enough.
- **Lost in ranking (1,014)** — similar name *and* address, yet outside the top 50 (many look-alikes). A larger K would recover part of these.
- **No address (961)** — only name keys are available; generic names are too common to block on.
- **Other (1,036)** — very short or heavily corrupted names/addresses.

---
# Summary — Phase 11

| kind | count (tuning) | F0.5 gain if fixed | biggest causes |
|---|---|---|---|
| A. false merges | 1,401 | +0.018 | same address + other name (often **Indian script**), same name + no address, near-duplicates |
| B. missed candidates | 4,853 | +0.026 | borderline scores, no address, **Indian script**, trade names |
| C. missed by blocking | 4,203 | +0.027 | **Indian script**, lost in ranking (top 50), no address |

**One cause shows up in all three kinds: names written in Indian scripts.** S1 names are always in English letters,
but ~20–28% of Indian S2/S3 names are in Devanagari, Tamil, Telugu, Kannada, Bengali, … — and they are
*transliterations* of the English name (`सिल्वर डेवलपर्स` = "silver developers"). Converting them back to Latin letters would let
name keys block them, give the name features something to compare, and let the model notice when the Hindi name is a *different* business.

**Proposed Phase 12 improvements, in order of expected value**
1. **Transliterate Indian-script text to Latin letters** (e.g. with the small `anyascii` library, ISC licence, offline — no external data)
   before blocking and features. Targets ~1,200 blocking misses, ~730 missed candidates and part of 322 false merges.
2. **More training data**: the model saw only 20,000 of the 1.77M training entities. More examples should mostly help the
   ~1,200 borderline cases and the rare patterns.
3. **Larger K** for blocking (100 instead of 50): targets part of the ~1,000 "lost in ranking" misses, at twice the scoring cost.

Each will be measured one at a time against **0.934** on the validation set, like in Phase 6.